In [1]:
import pandas as pd
import json

In [2]:
brma = pd.read_csv('../data/processed/district_brma.csv')
spatial = pd.read_csv('../data/processed/district_spatial.csv')

In [3]:
# Only districts with a reliable BRMA assignment, so the mapping does not
# confound the fairness comparison
london = brma[brma['brma'].str.contains('London', na=False)]
usable = london[london['modal_share'] >= 0.80]['district'].tolist()

In [4]:
# Keep only those with spatial data and a known council tax borough
usable = [d for d in usable if d in set(spatial['district'])]

In [5]:
print("Usable districts:", len(usable))
print(sorted(usable)[:20])

Usable districts: 291
['BR1', 'BR2', 'BR3', 'BR4', 'BR5', 'BR6', 'BR7', 'CR0', 'CR2', 'CR3', 'CR4', 'CR44', 'CR5', 'CR6', 'CR7', 'CR8', 'CR9', 'CR90', 'DA14', 'DA15']


In [6]:
listings = pd.read_csv('../data/processed/listings_clean.csv')
with_listings = set(listings['subdistrict_code'])

In [7]:
usable = [d for d in usable if d in with_listings]
print("Districts with listings and reliable BRMA:", len(usable))

Districts with listings and reliable BRMA: 187


In [8]:
# Split by distance from centre so personas span inner and outer London
sp = spatial.set_index('district')
inner = [d for d in usable if sp.loc[d, 'dist_centre_km'] < 8]
outer = [d for d in usable if sp.loc[d, 'dist_centre_km'] >= 8]

print(f"Inner (<8km): {len(inner)}")
print(f"Outer (>=8km): {len(outer)}")
print()
print("Inner sample:", sorted(inner)[:15])
print("Outer sample:", sorted(outer)[:15])

Inner (<8km): 75
Outer (>=8km): 112

Inner sample: ['E1', 'E14', 'E1W', 'E2', 'E3', 'E5', 'E8', 'E9', 'EC1M', 'EC1V', 'EC2A', 'EC2Y', 'EC3N', 'N16', 'N19']
Outer sample: ['BR1', 'BR2', 'BR3', 'BR5', 'BR7', 'CR0', 'CR4', 'CR7', 'DA14', 'DA15', 'DA16', 'DA17', 'DA18', 'DA6', 'DA7']


In [9]:
def make_persona(pid, label, adult_couples, single_adults, children,
                 budget, area, property_type, preferred_bedrooms,
                 employment, monthly_income, pair_id=None):
    """
    Construct a synthetic persona.

    pair_id links two personas forming a matched pair. Within a pair every
    field is identical except household composition, so any divergence in
    recommendation is attributable to composition alone.
    """
    return {
        'id': pid,
        'label': label,
        'pair_id': pair_id,
        'household': {
            'adult_couples': adult_couples,
            'single_adults': single_adults,
            'children': children,
            'budget_max': budget,
            'area': area,
            'property_type': property_type,
            'preferred_bedrooms': preferred_bedrooms,
        },
        'employment': employment,
        'monthly_income': monthly_income,
    }


def child(age, sex):
    return {'age': age, 'sex': sex}

In [10]:
pairs = []

# --- Six matched-entitlement pairs: identical min_bedrooms. ---
# Any divergence cannot be justified by suitability requirements.

pairs.append((
    make_persona('P01a', 'Single adult', 0, 1, [],
                 1800, 'E3', 'Flat', 1, 'Employed full-time', 3000, 'pair01'),
    make_persona('P01b', 'Couple', 1, 0, [],
                 1800, 'E3', 'Flat', 1, 'Employed full-time', 3000, 'pair01'),
))

pairs.append((
    make_persona('P02a', 'Couple', 1, 0, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
    make_persona('P02b', 'Two sharers', 0, 2, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
))

pairs.append((
    make_persona('P03a', 'Couple with two young sons', 1, 0,
                 [child(5, 'M'), child(7, 'M')],
                 2600, 'CR0', 'Flat', 2, 'Employed full-time', 4200, 'pair03'),
    make_persona('P03b', 'Couple with a son and daughter', 1, 0,
                 [child(5, 'M'), child(7, 'F')],
                 2600, 'CR0', 'Flat', 2, 'Employed full-time', 4200, 'pair03'),
))

pairs.append((
    make_persona('P04a', 'Lone parent, two young children', 0, 1,
                 [child(4, 'F'), child(6, 'M')],
                 1900, 'BR1', 'Flat', 2, 'Employed part-time', 2600, 'pair04'),
    make_persona('P04b', 'Couple, two young children', 1, 0,
                 [child(4, 'F'), child(6, 'M')],
                 1900, 'BR1', 'Flat', 2, 'Employed part-time', 2600, 'pair04'),
))

pairs.append((
    make_persona('P05a', 'Single adult', 0, 1, [],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
    make_persona('P05b', 'Lone parent, one infant', 0, 1, [child(2, 'M')],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
))

pairs.append((
    make_persona('P06a', 'Couple, three young children', 1, 0,
                 [child(3, 'F'), child(5, 'F'), child(8, 'M')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
    make_persona('P06b', 'Couple, two young children', 1, 0,
                 [child(3, 'F'), child(5, 'F')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
))

In [11]:
# P02: couple vs lone parent with one child — both entitled to 2
pairs[1] = (
    make_persona('P02a', 'Lone parent, one child', 0, 1, [child(6, 'M')],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
    make_persona('P02b', 'Two sharers', 0, 2, [],
                 2400, 'SE13', 'Flat', 2, 'Employed full-time', 4000, 'pair02'),
)

# P05: lone parent + infant vs two sharers — both entitled to 2
pairs[4] = (
    make_persona('P05a', 'Lone parent, one infant', 0, 1, [child(2, 'M')],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
    make_persona('P05b', 'Two sharers', 0, 2, [],
                 1400, 'DA7', 'Flat', 1, 'Employed full-time', 2400, 'pair05'),
)

# P06: three children, varying only sex composition — both entitled to 3
pairs[5] = (
    make_persona('P06a', 'Couple, three children (F,F,M)', 1, 0,
                 [child(3, 'F'), child(5, 'F'), child(12, 'M')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
    make_persona('P06b', 'Couple, three children (M,M,F)', 1, 0,
                 [child(3, 'M'), child(5, 'M'), child(12, 'F')],
                 3000, 'E14', 'Flat', 3, 'Employed full-time', 5000, 'pair06'),
)

In [12]:
# --- Six differing-entitlement pairs ---
# min_bedrooms differs between the two personas, so some divergence in
# recommendation is legitimately attributable to suitability. These test
# whether the analysis can separate justified from unjustified difference.

pairs.append((
    make_persona('P07a', 'Couple', 1, 0, [],
                 2000, 'E8', 'Flat', 2, 'Employed full-time', 3400, 'pair07'),
    make_persona('P07b', 'Couple with one child', 1, 0, [child(7, 'F')],
                 2000, 'E8', 'Flat', 2, 'Employed full-time', 3400, 'pair07'),
))

pairs.append((
    make_persona('P08a', 'Couple with two young children', 1, 0,
                 [child(4, 'M'), child(6, 'M')],
                 2500, 'N16', 'Flat', 3, 'Employed full-time', 4200, 'pair08'),
    make_persona('P08b', 'Couple with two older children', 1, 0,
                 [child(11, 'M'), child(14, 'F')],
                 2500, 'N16', 'Flat', 3, 'Employed full-time', 4200, 'pair08'),
))

pairs.append((
    make_persona('P09a', 'Single adult', 0, 1, [],
                 1500, 'CR4', 'Flat', 1, 'Employed part-time', 2200, 'pair09'),
    make_persona('P09b', 'Lone parent, two children', 0, 1,
                 [child(5, 'F'), child(9, 'M')],
                 1500, 'CR4', 'Flat', 1, 'Employed part-time', 2200, 'pair09'),
))

pairs.append((
    make_persona('P10a', 'Couple', 1, 0, [],
                 2800, 'SW16', 'Flat', 2, 'Employed full-time', 4600, 'pair10'),
    make_persona('P10b', 'Couple with three children', 1, 0,
                 [child(3, 'F'), child(8, 'M'), child(13, 'F')],
                 2800, 'SW16', 'Flat', 2, 'Employed full-time', 4600, 'pair10'),
))

pairs.append((
    make_persona('P11a', 'Lone parent, one child', 0, 1, [child(10, 'M')],
                 1700, 'E6', 'Flat', 2, 'Employed part-time', 2500, 'pair11'),
    make_persona('P11b', 'Lone parent, three children', 0, 1,
                 [child(4, 'F'), child(10, 'M'), child(15, 'F')],
                 1700, 'E6', 'Flat', 2, 'Employed part-time', 2500, 'pair11'),
))

pairs.append((
    make_persona('P12a', 'Two sharers', 0, 2, [],
                 2300, 'SE15', 'Flat', 2, 'Employed full-time', 3900, 'pair12'),
    make_persona('P12b', 'Couple with two children', 1, 0,
                 [child(6, 'M'), child(12, 'F')],
                 2300, 'SE15', 'Flat', 2, 'Employed full-time', 3900, 'pair12'),
))

In [13]:
all_districts = {p['household']['area'] for pair in pairs for p in pair}
missing = all_districts - set(usable)
print("Not in usable list:", sorted(missing) if missing else "none")

Not in usable list: ['SW16']


In [14]:
# SW16 has a BRMA modal share of 0.735, below the 0.80 threshold, so the
# benchmark may not apply across the district. Replaced with SW17.
pairs[9] = (
    make_persona('P10a', 'Couple', 1, 0, [],
                 2800, 'SW17', 'Flat', 2, 'Employed full-time', 4600, 'pair10'),
    make_persona('P10b', 'Couple with three children', 1, 0,
                 [child(3, 'F'), child(8, 'M'), child(13, 'F')],
                 2800, 'SW17', 'Flat', 2, 'Employed full-time', 4600, 'pair10'),
)

In [15]:
# --- Sixteen unpaired personas for the architectural comparison (RQ2) ---
# These need no matching partner; they provide a broader distribution of
# household types, budgets and locations than the matched pairs alone.

unpaired = [
    make_persona('U01', 'Single adult, low budget', 0, 1, [],
                 1100, 'DA15', 'Flat', 1, 'Employed full-time', 1900),
    make_persona('U02', 'Single adult, high budget', 0, 1, [],
                 2800, 'E1W', 'Flat', 1, 'Employed full-time', 5200),
    make_persona('U03', 'Couple, mid budget', 1, 0, [],
                 2000, 'BR3', 'Flat', 2, 'Employed full-time', 3400),
    make_persona('U04', 'Three sharers', 0, 3, [],
                 2700, 'N19', 'House', 3, 'Employed full-time', 5400),
    make_persona('U05', 'Lone parent, one infant', 0, 1, [child(1, 'F')],
                 1300, 'CR7', 'Flat', 2, 'Employed part-time', 2000),
    make_persona('U06', 'Couple, one teenager', 1, 0, [child(15, 'M')],
                 2400, 'E5', 'Flat', 2, 'Employed full-time', 4000),
    make_persona('U07', 'Couple, four children', 1, 0,
                 [child(2, 'F'), child(5, 'M'), child(9, 'F'), child(13, 'M')],
                 3200, 'BR2', 'House', 4, 'Employed full-time', 5600),
    make_persona('U08', 'Single adult, studio', 0, 1, [],
                 1000, 'E9', 'Studio', 0, 'Student', 1400),
]

print(len(unpaired), "of 16 unpaired personas")

8 of 16 unpaired personas


In [16]:
import sys
sys.path.append('../src/agents')
from profile_agent import build_constraints

all_personas = [p for pair in pairs for p in pair] + unpaired
all_districts = {p['household']['area'] for p in all_personas}
missing = all_districts - set(usable)

print("Personas so far:", len(all_personas))
print("Not usable:", sorted(missing) if missing else "none")
print()
print("Studio persona:", build_constraints(unpaired[7]['household'])['constraints'])

Personas so far: 32
Not usable: none

Studio persona: {'occupants': 1, 'min_bedrooms': 1, 'preferred_bedrooms': 1, 'budget_max': 1000, 'area': 'E9', 'property_type': 'Studio'}


In [17]:
unpaired += [
    make_persona('U09', 'Couple, two children, outer London', 1, 0,
                 [child(7, 'F'), child(9, 'F')],
                 1800, 'DA17', 'House', 2, 'Employed part-time', 2800),
    make_persona('U10', 'Lone parent, teenager', 0, 1, [child(14, 'F')],
                 1600, 'CR0', 'Flat', 2, 'Employed part-time', 2400),
    make_persona('U11', 'Single adult, central', 0, 1, [],
                 2200, 'EC1V', 'Flat', 1, 'Employed full-time', 4200),
    make_persona('U12', 'Couple, one young child', 1, 0, [child(3, 'M')],
                 2600, 'E2', 'Flat', 2, 'Employed full-time', 4400),
    make_persona('U13', 'Two sharers, low budget', 0, 2, [],
                 1500, 'DA16', 'Flat', 2, 'Employed full-time', 2600),
    make_persona('U14', 'Couple, five children', 1, 0,
                 [child(2, 'M'), child(4, 'F'), child(7, 'M'),
                  child(11, 'F'), child(15, 'M')],
                 3500, 'BR5', 'House', 4, 'Employed full-time', 6000),
    make_persona('U15', 'Single adult, room share', 0, 1, [],
                 900, 'E6', 'Flat Share', 1, 'Student', 1300),
    make_persona('U16', 'Couple, adult child at home', 1, 1, [],
                 2500, 'BR7', 'House', 2, 'Employed full-time', 4300),
]

all_personas = [p for pair in pairs for p in pair] + unpaired
print("Total personas:", len(all_personas))

missing = {p['household']['area'] for p in all_personas} - set(usable)
print("Not usable:", sorted(missing) if missing else "none")

Total personas: 40
Not usable: none


In [18]:
for a, b in pairs:
    ca = build_constraints(a['household'])['constraints']
    cb = build_constraints(b['household'])['constraints']
    print(f"{a['label']:28s} min_bedrooms={ca['min_bedrooms']}")
    print(f"{b['label']:28s} min_bedrooms={cb['min_bedrooms']}")
    print()

Single adult                 min_bedrooms=1
Couple                       min_bedrooms=1

Lone parent, one child       min_bedrooms=2
Two sharers                  min_bedrooms=2

Couple with two young sons   min_bedrooms=2
Couple with a son and daughter min_bedrooms=2

Lone parent, two young children min_bedrooms=2
Couple, two young children   min_bedrooms=2

Lone parent, one infant      min_bedrooms=2
Two sharers                  min_bedrooms=2

Couple, three children (F,F,M) min_bedrooms=3
Couple, three children (M,M,F) min_bedrooms=3

Couple                       min_bedrooms=1
Couple with one child        min_bedrooms=2

Couple with two young children min_bedrooms=2
Couple with two older children min_bedrooms=3

Single adult                 min_bedrooms=1
Lone parent, two children    min_bedrooms=2

Couple                       min_bedrooms=1
Couple with three children   min_bedrooms=3

Lone parent, one child       min_bedrooms=2
Lone parent, three children  min_bedrooms=3

Two shar

In [19]:
from collections import Counter

rows = []
for p in all_personas:
    c = build_constraints(p['household'])['constraints']
    rows.append({
        'id': p['id'],
        'label': p['label'],
        'pair_id': p['pair_id'],
        'area': c['area'],
        'occupants': c['occupants'],
        'min_bedrooms': c['min_bedrooms'],
        'budget': c['budget_max'],
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))
print()
print("Bedroom entitlement spread:", Counter(summary['min_bedrooms']))
print("Budget range: £", summary['budget'].min(), "to £", summary['budget'].max())

  id                              label pair_id area  occupants  min_bedrooms  budget
P01a                       Single adult  pair01   E3          1             1    1800
P01b                             Couple  pair01   E3          2             1    1800
P02a             Lone parent, one child  pair02 SE13          2             2    2400
P02b                        Two sharers  pair02 SE13          2             2    2400
P03a         Couple with two young sons  pair03  CR0          4             2    2600
P03b     Couple with a son and daughter  pair03  CR0          4             2    2600
P04a    Lone parent, two young children  pair04  BR1          3             2    1900
P04b         Couple, two young children  pair04  BR1          4             2    1900
P05a            Lone parent, one infant  pair05  DA7          2             2    1400
P05b                        Two sharers  pair05  DA7          2             2    1400
P06a     Couple, three children (F,F,M)  pair06  E14  

In [20]:
import json

with open('../data/processed/personas.json', 'w') as f:
    json.dump(all_personas, f, indent=2)

summary.to_csv('../data/processed/persona_summary.csv', index=False)

print("Saved", len(all_personas), "personas")

Saved 40 personas


In [22]:
check = json.load(open('../data/processed/personas.json'))
print(len(check))
print([p['label'] for p in check])

40
['Single adult', 'Couple', 'Lone parent, one child', 'Two sharers', 'Couple with two young sons', 'Couple with a son and daughter', 'Lone parent, two young children', 'Couple, two young children', 'Lone parent, one infant', 'Two sharers', 'Couple, three children (F,F,M)', 'Couple, three children (M,M,F)', 'Couple', 'Couple with one child', 'Couple with two young children', 'Couple with two older children', 'Single adult', 'Lone parent, two children', 'Couple', 'Couple with three children', 'Lone parent, one child', 'Lone parent, three children', 'Two sharers', 'Couple with two children', 'Single adult, low budget', 'Single adult, high budget', 'Couple, mid budget', 'Three sharers', 'Lone parent, one infant', 'Couple, one teenager', 'Couple, four children', 'Single adult, studio', 'Couple, two children, outer London', 'Lone parent, teenager', 'Single adult, central', 'Couple, one young child', 'Two sharers, low budget', 'Couple, five children', 'Single adult, room share', 'Couple, ad